# 人才保留風險分析：用分群打破平均值的假象
**吳育庭**　|　資料來源：IBM HR Analytics Employee Attrition Dataset（1,470 名員工、34 個變數）

整體離職率看起來是 16.1%，乍看沒什麼問題。但這份分析想知道的是：這個平均數字底下，是不是藏著某些被忽略的高風險群體？如果有，原因是什麼，可以怎麼處理？

## 1. 讀資料

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

df = pd.read_csv('cleaned_hr_employee_attrition.csv')
print(f'資料筆數: {df.shape[0]}　變數數: {df.shape[1]}')
df.head()

資料筆數: 1470　變數數: 34


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeNumber,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,Attrition_Num,OverTime_Num
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,2,Female,94,3,2,Sales Executive,4,Single,5993,19479,8,Yes,11,3,1,0,8,0,1,6,4,0,5,1,1
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,2,3,Male,61,2,2,Research Scientist,2,Married,5130,24907,1,No,23,4,4,1,10,3,3,10,7,1,7,0,0
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,4,4,Male,92,2,1,Laboratory Technician,3,Single,2090,2396,6,Yes,15,3,2,0,7,3,3,0,0,0,0,1,1
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,5,4,Female,56,3,1,Research Scientist,3,Married,2909,23159,1,Yes,11,3,3,0,8,3,3,8,7,3,0,0,1
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,7,1,Male,40,3,1,Laboratory Technician,2,Married,3468,16632,9,No,12,3,4,1,6,3,3,2,2,2,2,0,0


## 2. 先看一下資料乾不乾淨

開始分析前先檢查缺失值跟基本數值範圍，確認沒有奇怪的離群值再往下做。

In [2]:
missing = df.isnull().sum()
print('缺失值統計：')
print(missing[missing > 0] if missing.sum() > 0 else '無缺失值，資料完整。')

df[['Age', 'MonthlyIncome', 'YearsAtCompany', 'YearsSinceLastPromotion']].describe()

缺失值統計：
無缺失值，資料完整。


,Age,MonthlyIncome,YearsAtCompany,YearsSinceLastPromotion
count,1470.000000,1470.000000,1470.000000,1470.000000
mean,36.923810,6502.931293,7.008163,2.187755
std,9.135373,4707.956783,6.126525,3.222430
min,18.000000,1009.000000,0.000000,0.000000
25%,30.000000,2911.000000,3.000000,0.000000
50%,36.000000,4919.000000,5.000000,1.000000
75%,43.000000,8379.000000,9.000000,3.000000
max,60.000000,19999.000000,40.000000,15.000000


## 3. 整體離職率

先抓一個基準數字，後面所有的分群結果都要跟這個比，不然講「高」或「低」沒有意義。

In [3]:
overall_attrition = df['Attrition_Num'].mean()
print(f'全公司整體離職率：{overall_attrition:.1%}')
print(f'離職人數：{df["Attrition_Num"].sum()} / {len(df)}')

全公司整體離職率：16.1%
離職人數：237 / 1470


## 4. 拆到職位層級，看看平均數字背後在發生什麼

16.1% 是把所有職位混在一起算出來的。實際拆開來看，差異很大。

In [4]:
role_attrition = df.groupby('JobRole')['Attrition_Num'].agg(['mean', 'count']).sort_values('mean', ascending=False)
role_attrition.columns = ['離職率', '人數']
role_attrition['離職率'] = role_attrition['離職率'].apply(lambda x: f'{x:.1%}')
role_attrition

,離職率,人數
JobRole,,
Sales Representative,39.8%,83
Laboratory Technician,23.9%,259
Human Resources,23.1%,52
Sales Executive,17.5%,326
Research Scientist,16.1%,292
Manufacturing Director,6.9%,145
Healthcare Representative,6.9%,131
Manager,4.9%,102
Research Director,2.5%,80


Sales Representative 的離職率是 39.8%，是全公司平均的 2.5 倍，n=83 不算小樣本，這不是隨機波動。反過來，Manager 跟 Research Director 的離職率只有 2.5%–4.9%。

如果只看公司整體的 16.1%，根本看不出 Sales Representative 這邊正在大量流失人，HR 的資源投放方向應該優先往這裡看，不是平均分散到每個部門。

## 5. 那是什麼原因讓人想走？先一個一個變數看

我自己猜可能跟幾件事有關：薪水合不合理、工作負擔重不重、有沒有發展機會、跟主管的關係好不好。先一個一個拆開看，再決定要不要疊在一起看。

In [5]:
overtime_attrition = df.groupby('OverTime')['Attrition_Num'].mean()
print('OverTime 對離職率的影響：')
print(overtime_attrition.apply(lambda x: f'{x:.1%}'))
print(f'差距倍數：{overtime_attrition["Yes"] / overtime_attrition["No"]:.1f} 倍')

OverTime 對離職率的影響：
OverTime
No     10.4%
Yes    30.5%
Name: Attrition_Num, dtype: str
差距倍數：2.9 倍


有加班的人離職率 30.5%，沒加班的只有 10.4%，差快 3 倍，是目前看到影響最大的單一因素。而且這個如果要改善，成本相對低——檢視排班、調整人力配置就可以動手，不用等預算核准。

In [6]:
df['PromotionStale'] = df['YearsSinceLastPromotion'] >= 3
promo_attrition = df.groupby('PromotionStale')['Attrition_Num'].mean()
print('超過3年沒升遷，對離職率的影響：')
print(promo_attrition.apply(lambda x: f'{x:.1%}'))

超過3年沒升遷，對離職率的影響：
PromotionStale
False    17.0%
True     13.7%
Name: Attrition_Num, dtype: str


In [7]:
median_income = df['MonthlyIncome'].median()
df['LowIncome'] = df['MonthlyIncome'] < median_income
df['LowSatisfaction'] = df['JobSatisfaction'] <= 2

low_sat_low_income = df[(df['LowSatisfaction']) & (df['LowIncome'])]
print(f'工作滿意度低又薪資低於中位數的員工：n = {len(low_sat_low_income)}')
print(f'這群人的離職率：{low_sat_low_income["Attrition_Num"].mean():.1%}，是全公司平均的 {low_sat_low_income["Attrition_Num"].mean()/overall_attrition:.1f} 倍')

工作滿意度低又薪資低於中位數的員工：n = 285
這群人的離職率：26.7%，是全公司平均的 1.7 倍


## 6. 如果把風險因子疊在一起看，會不會更明顯？

單一變數的差異已經夠大了，但更想知道的是：如果一個人同時中了好幾個風險因子，離職率會不會疊加放大，還是只是各自獨立的小效果？

In [8]:
high_risk = df[(df['OverTime'] == 'Yes') & (df['PromotionStale'])]
low_risk = df[(df['OverTime'] == 'No') & (~df['PromotionStale'])]

print(f'加班+升遷停滯：n = {len(high_risk)}，離職率 = {high_risk["Attrition_Num"].mean():.1%}')
print(f'無加班+近期有升遷：n = {len(low_risk)}，離職率 = {low_risk["Attrition_Num"].mean():.1%}')
print(f'差距倍數：{high_risk["Attrition_Num"].mean() / low_risk["Attrition_Num"].mean():.1f} 倍')

加班+升遷停滯：n = 100，離職率 = 28.0%
無加班+近期有升遷：n = 781，離職率 = 11.1%
差距倍數：2.5 倍


果然有疊加效果。同時加班又升遷停滯的人，離職率28%，差不多是「正常」那群人的2.5倍。這種人如果只看公司整體報表，完全不會被注意到——他們可能分散在好幾個部門裡，個別部門的數字看起來都還算正常。

## 7. 把這些因子組合成一個風險分數

四個因子單獨看都有效果，但如果要讓HR實際拿來排序「誰該先處理」，最後還是要有一個單一指標。做法是：先看每個因子單獨造成的離職率差異有多大，差異大的給高一點的權重，組合成一個分數。

這個邏輯之後會搬到 Power BI 用 DAX 重做一次，這裡先用 Python 驗證權重設定合不合理。

In [9]:
def calc_risk_score(row):
    risk_overtime = 1 if row['OverTime'] == 'Yes' else 0
    risk_promo = 1 if row['YearsSinceLastPromotion'] >= 3 else 0
    risk_satisfaction = 1 if row['JobSatisfaction'] <= 2 else 0
    risk_income = 1 if row['MonthlyIncome'] < median_income else 0

    return (risk_overtime * 0.35 +
            risk_promo * 0.25 +
            risk_satisfaction * 0.25 +
            risk_income * 0.15)

df['Weighted_Risk_Score'] = df.apply(calc_risk_score, axis=1)

df['Risk_Tier'] = pd.cut(df['Weighted_Risk_Score'],
                          bins=[-0.01, 0.25, 0.5, 1.0],
                          labels=['低風險', '中風險', '高風險'])

risk_validation = df.groupby('Risk_Tier')['Attrition_Num'].agg(['mean', 'count'])
risk_validation.columns = ['實際離職率', '人數']
risk_validation['實際離職率'] = risk_validation['實際離職率'].apply(lambda x: f'{x:.1%}')
risk_validation

,實際離職率,人數
Risk_Tier,,
低風險,8.4%,715
中風險,20.5%,513
高風險,29.8%,242


分數越高，實際離職率也確實越高，三層之間拉得開，代表這個權重設定至少在方向上是對的。實際做成 Power BI dashboard 後，HRBP可以直接用這個分數排序，決定先去找誰談。

## 8. 這樣值多少錢

光講離職率的差異，對管理層來說還是太抽象，最後要換算成數字。

In [10]:
sales_rep = df[df['JobRole'] == 'Sales Representative']
current_rate = sales_rep['Attrition_Num'].mean()
target_rate = overall_attrition
avg_annual_income = sales_rep['MonthlyIncome'].mean() * 12

current_leavers = len(sales_rep) * current_rate
target_leavers = len(sales_rep) * target_rate
reduction = current_leavers - target_leavers

print(f'Sales Representative 現況：離職率 {current_rate:.1%}，每年約 {current_leavers:.0f} 人離職')
print(f'如果能降到全公司平均 {target_rate:.1%}，每年離職人數約 {target_leavers:.0f} 人')
print(f'差距：每年少流失約 {reduction:.0f} 人')
print()
print('用不同的離職成本假設換算一下：')
for mult in [0.5, 1.0, 1.5]:
    saving = reduction * avg_annual_income * mult
    print(f'  假設離職成本 = {mult}x 年薪 → 每年大概省下 {saving:,.0f}（資料集薪資單位）')

Sales Representative 現況：離職率 39.8%，每年約 33 人離職
如果能降到全公司平均 16.1%，每年離職人數約 13 人
差距：每年少流失約 20 人

用不同的離職成本假設換算一下：
  假設離職成本 = 0.5x 年薪 → 每年大概省下 309,107（資料集薪資單位）
  假設離職成本 = 1.0x 年薪 → 每年大概省下 618,214（資料集薪資單位）
  假設離職成本 = 1.5x 年薪 → 每年大概省下 927,321（資料集薪資單位）


這裡用的是業界常見的離職成本估算法（招募+訓練+生產力空窗，約年薪的0.5–1.5倍），實際數字要看公司自己的招募成本結構，這裡只是給個量級參考。

## 9. 這份分析的限制

老實說幾個地方還不夠嚴謹：

- 這是橫斷面資料，只能看相關性、排風險高低，不能說「加班導致離職」這種因果關係
- 權重是憑單變數的差異幅度設的，比較直覺，如果要更嚴謹應該用邏輯回歸或隨機森林重新驗證一次
- 真要落地的話，建議公司開始做結構化的離職面談，累積追蹤資料，才能真的驗證行動方案有沒有效
- 財務效益那段是示意性的，真實案子要用客戶自己的薪資水準跟招募成本重新算

## 10. 接下來可以做的事

| 時間軸 | 做什麼 |
|---|---|
| 現在就能動手（0–3個月） | 看 Sales Representative 跟其他高加班部門的人力配置、排班方式 |
| 中期（3–12個月） | 建立晉升時間表跟內部薪酬檢視機制 |
| 長一點（12個月以上） | 開始做結構化離職面談，累積資料 |
